# Animate Migration Tracks + ERA5 Weather

Test notebook for `animate_tracks_weather.py`.  
**Kernel**: `migration` conda env  
**Data**: `move_icarus_bats_solarnoon_daily.csv` — filtered to April 2026

### CSV loading notes
The file is exported from R with `write.csv()`.  Non-location rows (VeDBA only) have
`c(NA, NA)` in the geometry column — an unquoted comma that adds an extra field and shifts
all subsequent columns on those rows.  We work around this with `engine='python',
on_bad_lines='skip'` and use the pre-extracted `lon`/`lat` columns.  
Event timestamps live in `start_timestamp` for location rows (`timestamp` is NA for those).

In [5]:
import sys, subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from python.animate_tracks_weather import animate_tracks_weather

CSV      = Path(r"C:/Users/Edward/Dropbox/MPI/Noctule/Data/rdata/move_icarus_bats_solarnoon_daily.csv")
# UNC path must use backslashes on Windows for Python to resolve it correctly
ERA5_DIR = Path(r"\\10.0.16.7\grpdechmann\Postdoc-EdwardHurme\EnvData")
OUT_FILE = "migration_april2026.gif"

print("CSV exists:     ", CSV.exists())
print("ERA5 dir exists:", ERA5_DIR.exists())
print("single_levels/: ", (ERA5_DIR / "single_levels").exists())

CSV exists:      False
ERA5 dir exists: False
single_levels/:  False


In [2]:
# Install cfgrib if not already present (needed to read .grib files)
# Run once, then restart kernel
subprocess.run([sys.executable, "-m", "pip", "install", "cfgrib"], check=True)

CompletedProcess(args=['C:\\Users\\Edward\\miniconda3\\envs\\gee\\python.exe', '-m', 'pip', 'install', 'cfgrib'], returncode=0)

In [3]:
# ── Load CSV ──────────────────────────────────────────────────────────────────
# engine='python' + on_bad_lines='skip' silently drops the ~362k VeDBA-only rows
# whose geometry column has unquoted c(NA, NA) (adds an extra comma, breaking the row).
# The remaining ~57k rows are location fixes with real lon/lat values.
df_raw = pd.read_csv(CSV, index_col=0, engine='python', on_bad_lines='skip')

# Keep only rows with actual coordinates
df_loc = df_raw.dropna(subset=['lon', 'lat']).copy()

# Event time is in start_timestamp for location rows (timestamp col is NA here)
df_loc['ts'] = pd.to_datetime(df_loc['start_timestamp'], errors='coerce', utc=True)

print(f"Total location rows: {len(df_loc)}")
print(f"Date range:          {df_loc['ts'].min().date()} – {df_loc['ts'].max().date()}")
print(f"Years:               {sorted(df_loc['ts'].dt.year.dropna().unique().astype(int))}")

Total location rows: 57405
Date range:          2022-04-14 – 2026-05-06
Years:               [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


## Filter to April 2026

In [4]:
df_april = df_loc[
    (df_loc['ts'].dt.year  == 2026) &
    (df_loc['ts'].dt.month == 4)
].copy()

# Drop original broken columns before renaming to avoid duplicate column names
df_april = df_april.drop(columns=['timestamp', 'individual_local_identifier'], errors='ignore')

df_april = df_april.rename(columns={
    'lon': 'location_long',
    'lat': 'location_lat',
    'ts':  'timestamp',
    'tag_local_identifier': 'individual_local_identifier',
})

print(f"April 2026 fixes: {len(df_april)}")
print(f"Date range:       {df_april['timestamp'].min().date()} – {df_april['timestamp'].max().date()}")
print(f"lon: {df_april['location_long'].min():.1f} – {df_april['location_long'].max():.1f}")
print(f"lat: {df_april['location_lat'].min():.1f} – {df_april['location_lat'].max():.1f}")
print(f"\nIndividuals ({df_april['individual_local_identifier'].nunique()}):")
summary = (df_april.groupby(['individual_local_identifier', 'species'])
           .size().reset_index(name='n').sort_values('individual_local_identifier'))
print(summary.to_string(index=False))

April 2026 fixes: 4761
Date range:       2026-04-01 – 2026-04-30
lon: -7.6 – 17.3
lat: 39.5 – 51.8

Individuals (93):
individual_local_identifier              species   n
                    120D5CB    Nyctalus leisleri 135
                    120D5D3     Nyctalus noctula  36
                    120D5E2    Nyctalus leisleri 115
                    120D5E3    Nyctalus leisleri  50
                    120D675     Nyctalus noctula  68
                    120D68D     Nyctalus noctula   8
                    120D723     Nyctalus noctula   9
                    120D731    Nyctalus leisleri  36
                    120D7C9    Nyctalus leisleri  47
                    120D986     Nyctalus noctula  18
                    120D990     Nyctalus noctula  11
                    120D9CB     Nyctalus noctula  13
                    120D9CE     Nyctalus noctula  15
                    1212DF1     Nyctalus noctula   7
                    12132C2    Nyctalus leisleri  24
                     9E6A77     Ny

In [5]:
# ── Static preview ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
species_colors = {'Nyctalus noctula': '#e41a1c', 'Nyctalus leisleri': '#377eb8',
                  'Nyctalus lasiopterus': '#4daf4a'}

for ind, grp in df_april.groupby('individual_local_identifier'):
    grp_s = grp.sort_values('timestamp')
    sp = grp_s['species'].iloc[0] if 'species' in grp_s.columns else 'unknown'
    c = species_colors.get(sp, '#999999')
    ax.plot(grp_s['location_long'], grp_s['location_lat'],
            '-', color=c, linewidth=0.6, alpha=0.6)
    ax.scatter(grp_s['location_long'].iloc[-1], grp_s['location_lat'].iloc[-1],
               color=c, s=20, zorder=3)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=v, label=k) for k, v in species_colors.items()], fontsize=8)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('April 2026 — all tracks (dot = last fix)')
plt.tight_layout()
plt.show()

<positron-console-cell-5>:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Check ERA5 coverage for April 2026

In [6]:
single_dir = ERA5_DIR / "single_levels"
print(f"single_levels/ exists: {single_dir.exists()}")

if single_dir.exists():
    files = sorted(single_dir.glob("era5_single_*"))
    print(f"\nAvailable files ({len(files)}):")
    for f in files:
        print(" ", f.name)

    # Accept either .nc or .grib
    needed_stems = ["era5_single_2026_04"]
    for stem in needed_stems:
        found = any((single_dir / f"{stem}{ext}").exists() for ext in (".nc", ".grib"))
        status = "✓" if found else "✗ MISSING — run download_era5.py for 2026/April"
        print(f"\n{status}  {stem}.*")
else:
    print("⚠  ERA5 directory not reachable")

single_levels/ exists: True

Available files (56):
  era5_single_2022_01.grib
  era5_single_2022_02.grib
  era5_single_2022_03.grib
  era5_single_2022_04.grib
  era5_single_2022_05.grib
  era5_single_2022_06.grib
  era5_single_2022_07.grib
  era5_single_2022_08.grib
  era5_single_2022_09.grib
  era5_single_2022_10.grib
  era5_single_2022_11.grib
  era5_single_2022_12.grib
  era5_single_2023_01.grib
  era5_single_2023_02.grib
  era5_single_2023_03.grib
  era5_single_2023_04.grib
  era5_single_2023_05.grib
  era5_single_2023_06.grib
  era5_single_2023_07.grib
  era5_single_2023_08.grib
  era5_single_2023_09.grib
  era5_single_2023_10.grib
  era5_single_2023_11.grib
  era5_single_2023_12.grib
  era5_single_2024_01.grib
  era5_single_2024_02.grib
  era5_single_2024_03.grib
  era5_single_2024_04.grib
  era5_single_2024_05.grib
  era5_single_2024_06.grib
  era5_single_2024_07.grib
  era5_single_2024_08.grib
  era5_single_2024_09.grib
  era5_single_2024_10.grib
  era5_single_2024_11.grib
  er

## Render animation

95 individuals, 4 761 fixes, Iberia → central Europe.  
Start with a fast draft (12h frames) to check layout, then uncomment the full render.

In [ ]:
# ── Draft render (~60 frames at 12h) ─────────────────────────────────────────
animate_tracks_weather(
    df_april,
    era5_dir=ERA5_DIR,
    out_file="migration_april2026_draft.gif",

    time_col="timestamp",
    lon_col="location_long",
    lat_col="location_lat",
    id_col="individual_local_identifier",
    color_col="species",            # colour tracks by species, not individual

    show_pressure=True,
    show_temperature=True,
    show_precipitation=False,
    show_wind_barbs=False,

    isobar_interval=4,
    isobar_bold_interval=20,
    scale_bar_km=500,

    time_resolution_hours=12,
    fps=6,
    width_px=900,
    height_px=620,
    dpi=100,
)

Loading tracks …
  4761 fixes | 93 individual(s) | 2026-04-01 – 2026-04-30
Loading ERA5 single-level data …
  Loading 3 ERA5 file(s) (cfgrib) …


In [ ]:
# ── Full-quality render — uncomment after draft looks good ────────────────────
# animate_tracks_weather(
#     df_april,
#     era5_dir=ERA5_DIR,
#     out_file=OUT_FILE,
#
#     time_col="timestamp",
#     lon_col="location_long",
#     lat_col="location_lat",
#     id_col="individual_local_identifier",
#
#     show_pressure=True,
#     show_temperature=True,
#     show_precipitation=True,
#     show_wind_barbs=True,
#
#     isobar_interval=4,
#     isobar_bold_interval=20,
#     wind_density=8,
#
#     time_resolution_hours=6,
#     fps=8,
#     width_px=1400,
#     height_px=900,
#     dpi=120,
# )